# Phase 2 — per-design 3D topology optimization of the top-10 blades

Runs **frozen-aero-skin SIMP TO on each** of the top-10 designs (ranked by the fine 3D
`J_fan` from Stage-3.C), trimming mass by hollowing the rib cores + thick-panel interiors
while holding the air-pushing panel faces solid. Output: a carved density field per design +
a `summary.json` with the mass removed and the structural screen (tip deflection, von Mises).
Then choose **3** of the 10 to print.

Logic lives in `fanopt.topopt.blade_topopt` + `scripts/run_phase2_blade_to.py`; this notebook
only orchestrates. Each design is ~10-15 min at the 1.5 mm mesh, so the batch is ~2-3 h.


## 1. Connect Drive

In [ ]:
# Drive connect ONLY (kept separate from the repo/deps install below).
import importlib.util
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/fanopt")
else:
    DRIVE_ROOT = Path.cwd() / "data"
print("drive root:", DRIVE_ROOT)

## 2. Repo + deps  (separate cell from the Drive connect above)

In [ ]:
import importlib.util, os, subprocess, sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
BRANCH = "main"  # the TO tool + this notebook land on main
REPO = Path("/content/fan-optimization") if IN_COLAB else Path.cwd()
if IN_COLAB:
    if not REPO.exists():
        subprocess.run(["git", "clone", "-b", BRANCH,
                        "https://github.com/clingergab/fan-optimization.git", str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "pull", "origin", BRANCH], check=True)
    subprocess.run("apt-get install -qq -y libglu1-mesa libxrender1 libxcursor1 "
                   "libxft2 libxinerama1 unzip".split(), check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[bo]"], check=True)
    # TO stack: gmsh + CadQuery for the solid mesh, scikit-fem for the 3D FEA.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "gmsh", "cadquery", "scikit-fem"], check=True)
for p in (str(REPO), str(REPO / "src"), str(REPO / "scripts")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("repo:", REPO)

## 3. Config — point at the campaign + its Stage-3.C verification

In [ ]:
from fanopt.topopt.blade_topopt import DEFAULT_VOLFRAC
from fanopt.topopt.blade_fea_mesh import FeaMeshParams

SHARED_DIR   = DRIVE_ROOT / "campaign_trapezoid"                      # the campaign shards
VERIFICATION = DRIVE_ROOT / "phase5_verify_blade" / "verification.json"  # Stage-3.C fine J_fan
OUT_DIR      = DRIVE_ROOT / "phase2_blade_to"                         # carved fields + summary (Drive)
TOP_K        = 10          # TO the top-10 by fine J_fan (then pick 3 to print)
VOLFRAC      = DEFAULT_VOLFRAC     # retained share of the carvable region (0.40)
MAX_ITERS    = 40
MESH_SIZE_M  = FeaMeshParams().mesh_size_m   # 1.5 mm production mesh

assert SHARED_DIR.exists(), f"campaign folder not found: {SHARED_DIR}"
assert VERIFICATION.exists(), f"verification.json not found: {VERIFICATION} (run Stage-3.C first)"
print(f"campaign: {SHARED_DIR}")
print(f"verify:   {VERIFICATION}")
print(f"top-{TOP_K} | volfrac {VOLFRAC} | {MAX_ITERS} iters | mesh {MESH_SIZE_M*1e3:.1f} mm | out {OUT_DIR}")

## 4. Preview — the top-10 designs that will be TO'd (by fine J_fan)

In [ ]:
from fanopt.cfd.blade_verify import top_verified_designs

designs = top_verified_designs(SHARED_DIR, VERIFICATION, top_k=TOP_K)
print(f"{len(designs)} designs to optimize (best fine J_fan first):")
for name, params, j3d in designs:
    print(f"  {name}   J_fan_3d={j3d:.4g}   rib_mode={'uniform' if params.uniform else 'ribbed'}")

## 5. RUN the per-design 3D TO
Writes `<name>_density.npy` + `summary.json` under `OUT_DIR` (on Drive). A failed design is
recorded with an `error` and skipped — one bad blade never aborts the batch.

In [ ]:
import run_phase2_blade_to

summary = run_phase2_blade_to.run(
    shared_dir=SHARED_DIR,
    verification=VERIFICATION,
    out_dir=OUT_DIR,
    top_k=TOP_K,
    volfrac=VOLFRAC,
    max_iters=MAX_ITERS,
    mesh_size_m=MESH_SIZE_M,
    skin_thickness_m=None,   # default shell sized to the mesh
    progress=True,
)
print(f"\n{summary['n_succeeded']}/{summary['n_designs']} designs TO'd -> {OUT_DIR}/summary.json")

## 6. Results — mass removed + structural screen per design

In [ ]:
import json
from fanopt.geometry.schema import SIGMA_Y_PETG_Z_PA

summ = json.loads((OUT_DIR / "summary.json").read_text())
rows = [r for r in summ["designs"] if "error" not in r]
rows.sort(key=lambda r: r["volume_removed_frac"], reverse=True)
yield_mpa = SIGMA_Y_PETG_Z_PA / 1e6

print(f"{'name':22} {'removed%':>8} {'mass_g':>7} {'u_tip_mm':>9} {'VM_MPa':>7}  screen")
for r in rows:
    ok = (r["u_tip_max_mm"] < 1.0) and (r["max_von_mises_mpa"] < yield_mpa)
    print(f"{r['name']:22} {r['volume_removed_frac']*100:7.1f} {r['mass_kg']*1e3:7.1f} "
          f"{r['u_tip_max_mm']:9.3f} {r['max_von_mises_mpa']:7.2f}  {'PASS' if ok else 'CHECK'}")
for r in summ["designs"]:
    if "error" in r:
        print(f"  [error] {r['name']}: {r['error']}")
print(f"\nscreen = tip deflection < 1 mm AND max von Mises < {yield_mpa:.0f} MPa (PETG weak-axis yield)")
print("NOTE: this is the Phase-2 screen; the binding cert is the §59.5 combined-blade gate.")

## 7. Render carved vs solid — top designs in 3D
Loads the element centroids saved alongside each density field (no re-meshing, so the
colouring is guaranteed to align) and shows retained material — solid shell + retained core.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

N_SHOW = min(3, len(designs))
fig = make_subplots(rows=1, cols=N_SHOW, specs=[[{"type": "scene"}] * N_SHOW],
                    subplot_titles=[d[0][:14] for d in designs[:N_SHOW]])
for col, (name, _params, _j) in enumerate(designs[:N_SHOW], start=1):
    dens = np.load(OUT_DIR / f"{name}_density.npy")
    cen = np.load(OUT_DIR / f"{name}_centroids.npy")   # saved by the batch -> aligned to dens
    keep = dens > 0.5                                    # material retained after TO
    fig.add_trace(go.Scatter3d(
        x=cen[keep, 0], y=cen[keep, 1], z=cen[keep, 2], mode="markers",
        marker=dict(size=1.5, color=dens[keep], colorscale="Viridis", cmin=0.5, cmax=1.0),
        showlegend=False), row=1, col=col)
fig.update_layout(height=460, width=320 * N_SHOW, title="Retained material (density > 0.5)")
fig.show()